In [ ]:
from nn import *
from ray import train, tune
from ray.tune.search.hyperopt import HyperOptSearch


def Mix(i, rqh=0, rkh=0, rk=0, rvh=0, rv=0, roh=0, cch=0, cc=0, coh=0, ah=0, mh=0, sh=0, h=0, oh=0, j=0, o=0, rh=1, cg=1, rr=1, cr=1, ar=1, mr=1, b=False):
    h = h or i
    o = o or h
    return Add(
        Seq(
            Add(
                Ret(i, rqh, rkh, rh, rvh, rv, roh, h, rh, rr, min=1, max=8),
                SepCovLeft(i, cch, cc, coh, h, cg, cr),
                *[ApLeft(i, ah, h, ar)] if ah else [],
                *[MpLeft(i, mh, h, mr)] if mh else [],
                *[SeLeft(i, sh, h)] if sh else [],
            ),
            Bn(h),
            SiLU(True),
            Sep(h, oh, o),
        ),
        Sep(i, j, o, b),
    )


def Mixs(e, m, **_):
    return Seq(
        Embedding(e, m[0]["i"]),
        *[
            Mix(
                **dict(zip(m, M)),
                o=dict(zip(m, o))["i"] if b or B else e,
                b=not (b or B),
            )
            for b, m, M in pk(m)
            for B, M, o in pk(np.linspace(*[m.pop("l", 1) + 1, [M[m] for m in m], list(m.values())][::-1], dtype=int).tolist())
        ],
    )


def train_mixs(O, dtype=torch.bfloat16, compile=lambda m, x: [print(m, file=open("/tmp/m", "a")), torch.compile(m)][1], report=train.report):
    a = Dat("py.txt", 32, 128)
    X = a().to("cuda", torch.long)
    m = bisect(lambda x: Mixs(**mul(O, x)), size, 64, O.get("sz", 10000000))[1]
    m = compile(m.to("cuda", dtype), X)
    o = torch.optim.Adam(m.parameters(), **O.get("o", {}))
    O = torch.optim.lr_scheduler.OneCycleLR(o, **dict(max_lr=0.01, total_steps=10000, pct_start=0.01, div_factor=3, final_div_factor=9) | O.get("O", {}))
    L = A = 0
    for _ in range(1, 1 + O.total_steps):
        x, X = X, a().to("cuda", torch.long)
        y = m(x[:, :-1])
        x = x[:, 1:]
        l = functional.cross_entropy(y.mT, x)
        l.backward()
        o.step()
        o.zero_grad()
        O.step()
        L += l.data.log()
        A += (y.argmax(2) != x).sum().log()
        if not _ % 100:
            report({"l": float((L / 100).exp()), "a": 1 - float((A / 100).exp()) / x.numel()})
            L = A = 0


a = tune.Tuner.restore("ray_results/train_mixs_2023-11-01_17-25-38/", train_mixs).get_results()

In [ ]:
O = a.get_best_result().config
m = bisect(lambda x: Mixs(**mul(O, x)), size, 64, O.get("sz", 10000000), log=print)[1]

In [ ]:
m

In [ ]:
def flat_(a, p=()):
    if isinstance(a, dict):
        for i, a in a.items():
            yield from flat_(a, p + (i,))
    elif isinstance(a, (list, tuple)):
        for a in a:
            yield from flat_(a, p)
    elif isinstance(a, float):
        yield p, a


def flat(a):
    t = {}
    for i, a in flat_(a):
        t.setdefault(i, []).append(a)
    return {i: sum(a) / len(a) for i, a in t.items()}


for i, (l, r) in (
    pd.DataFrame([flat(a.config) for a in sorted(a, key=lambda a: -a.metrics.get("a", 0))[: len(a) // 10]])
    .agg(
        (
            lambda x: x.mean() - 2 * x.std(),
            lambda x: x.mean() + 2 * x.std(),
        )
    )
    .T.iterrows()
):
    print(f"'{i[-1]}':tune.loguniform({l:.1f},{r:.1f}),")

In [ ]:
def str_model(a, l=0, m={Sequential: (0, " "), Add: (1, "+"), SiLU: ""}):
    if isinstance(a, Sequential):
        _ = m[type(a)][1].join(_ for _ in a for _ in [str_model(_, m[type(a)][0])] if _ != None)
        return _ if m[type(a)][0] >= l else "(" + _ + ")"
    if isinstance(a, Conv1d):
        return f"{a.in_channels}" + (f"_{a.kernel_size[0]}" if a.kernel_size[0] > 1 else "")
    return m.get(type(a))


for i in sorted(a, key=lambda x: -x.metrics.get("a", 0))[:20]:
    print("%.2f" % i.metrics["l"], "%.4f" % ((np.log1p(-1 / 68) - np.log1p(-i.metrics["a"] / 100)) / np.log(2)), str_model(get_model(i.config)))

```
259.50 1.3729 (48 28_11 28  48 10)+48 (37_3  183)+37 (37_3  184)+37 (38_3  185)+38 39
261.61 1.3726 (43 30_11 30  62 10)+43 (35_3  171)+35 (35_3  196)+35 (36_3  224)+36 37
262.19 1.3719 (53 32_11 32  74 9)+53 (33_3  177)+33 (33_3  204)+33 (34_3  236)+34 35
261.66 1.3719 (50 24_11 24  79 11)+50 (26_3  242)+26 (28_3  247)+28 (31_3  253)+31 34
261.11 1.3718 (57 49_11 49  93 8)+57 (32_3  194)+32 (35_3  180)+35 (38_3  167)+38 42
260.05 1.3716 (63 41_11 41  91 12)+63 (27_3  205)+27 (31_3  199)+31 (37_3  194)+37 44
262.42 1.3715 (47 34_11 34  68 9)+47 (28_3  209)+28 (32_3  208)+32 (37_3  207)+37 43
261.58 1.3713 (56 23 61_11 61  103 8)+56 (26_3  203)+26 (30_3  200)+30 (35_3  197)+35 42
262.45 1.3713 (49 29_9 29  74 8)+49 (26_4  199)+26 (30_3  208)+30 (36_3  218)+36 44
261.98 1.3711 (57 56_11 56  93 8)+57 (39_3  184)+39 (37_3  170)+37 (36_3  158)+36 36
261.48 1.3711 (53 42_11 42  96 8)+53 (30_3  175)+30 (33_3  197)+33 (36_3  221)+36 40
260.55 1.3710 (45 32_10 32  41 7)+45 (32_3  251)+32 (34_3  206)+34 (36_3  170)+36 39
260.64 1.3710 (57 43_11 43  83 10)+57 (28_3  180)+28 (31_3  201)+31 (36_3  226)+36 41
261.06 1.3708 (46 31_10 31  57 7)+46 (26_4  216)+26 (31_3  202)+31 (37_3  189)+37 45
261.97 1.3707 (55 37_11 37  97 10)+55 (31_3  175)+31 (34_3  188)+34 (37_3  203)+37 41
262.48 1.3707 (55 43_11 43  93 8)+55 (33_3  212)+33 (35_3  185)+35 (37_3  162)+37 40
262.67 1.3706 (56 24_11 24  90 15)+56 (34_3  134)+34 (34_3  192)+34 (35_3  275)+35 36
260.78 1.3703 (57 31_11 31  74 8)+57 (41_3  199)+41 (39_3  174)+39 (38_3  152)+38 37
261.16 1.3702 (47 30_11 30  71 7)+47 (29_3  261)+29 (31_3  218)+31 (34_3  183)+34 38
261.92 1.3701 (63 37_11 37  99 9)+63 (39_3  211)+39 (36_3  184)+36 (33_3  160)+33 31

261.09 1.3744 (62 25_2  271)+(62 15) (46_4  159)+46 (35 45_12 45  94)+35 (26_9  49)+26_3 43
261.33 1.3735 (74 12_2  144)+(74 13) (52_5  125)+52 (37 95_17 95  109)+37 (58 98_13 98  33 16)+58_2 34
261.23 1.3713 (83_2 28  136)+(83 9) (48_5  146)+48 (28 48_16 48  157)+28 (42 91_13 91  38 12)+42_2 63
261.73 1.3708 (80 16_2  92)+(80 6) (49_5  134)+49 (31 75_14 75  197)+31 (47 80_14 80  28 14)+47_2 52
264.61 1.3702 (65 21_2  111)+(65 17) (51_5  121)+51 (41 62_13 62  133)+41_2 (42 125_6 125  60 12)+42_3 43
264.23 1.3698 (88 14_2  95)+(88 10) (51_5  126)+51 (30 98_13 98  168)+30 (43 65_13 65  42 22)+43_2 53
264.05 1.3695 (84 17_2 24  161)+(84 10) (62_5  125)+62 (47 57_16 57  98)+47 (38 93_16 93  25)+38_2 37
263.47 1.3694 (85 11_2  100)+(85 6) (47_4  129)+47 (27 93_12 93  168)+27 (42 78_15 78  76 25)+42_2 89
262.78 1.3666 (59 12_2  72)+(59 7) (57 126_5 126  155)+57 (56 27_15 27  334)+56 (54 53_18 53  51 12)+54_2 50
263.98 1.3661 (64 13_3  389)+(64 19) (48 28_6  152)+48 (37 56_12 56  60)+37_2 (21_6  72)+21 41
265.48 1.3649 (78 18_3  218)+(78 18) (57 33_6  125)+57 (42 89_16 89  72)+42_2 (17_7  75)+17 50
266.86 1.3647 (74 9_2  125)+(74 10) (51_5  119)+51 (36 146_15 146  114)+36 (32 95_13 95  60 16)+32_2 52
263.81 1.3645 (92 20_2  124)+(92 11) (58_5  116)+58 (37 66_14 66  110)+37_2 (39 143_13 143  36 17)+39_2 37
264.91 1.3642 (87 31_2  222)+(87 14) (48_5  113)+48 (27 68_13 68  58)+27_2 (36_9  46)+36_3 32
264.25 1.3638 (86 19_2  122)+(86 11) (49_5  135)+49 (29 69_17 69  150)+29 (35_13  28 13)+35_2 51
263.81 1.3637 (90 12_2 19  162)+(90 8) (55_4  153)+55 (34 58_12 58  145)+34 (45 83_12 83  25 9)+45_2 66
263.7 1.3635 (90_2 25  94)+(90 13) (62_4  113)+62 (43 79_11 79  137)+43 (41 88_15 88  42 8)+41_2 74
265.95 1.3619 (86 15_3  262)+(86 20) (60 37_6  87)+60 (43 112_15 112  29)+43_2 (31_7  53)+31 60
263.81 1.3612 (80 27_2 25  82)+(80 8) (44_4  143)+44 (25 44_12 44  252)+25 (51 125_16 125  25 15)+51_2 77
266.7 1.3607 (78 26_2 24  107)+(78 14) (45_6  130)+45 (26 106_18 106  160 29)+26 (42 44_14 44  28 5)+42_2 59
```

In [ ]:
import pandas as pd

for i, (l, r) in (
    np.exp(
        np.log(pd.DataFrame({i: [a[i] or 1 for a in sorted(a, key=lambda x: -x.metrics["a"])[: len(a) // 10] for a in a.config["l"]] for i in "ihctauHCTrRl"})).agg(
            [
                lambda i: i.mean() - 2 * i.std(),
                lambda i: i.mean() + 2 * i.std(),
            ]
        )
    )
    .round(1)
    .T.iterrows()
):
    print(f"{i}=tune.loguniform({l},{r}){'if i else 1.'if i=='i'else''},")

In [ ]:
from alphashape import alphashape
from matplotlib import pyplot as plt


def lower(x):
    min = (x[:, 0] + x[:, 1] * 1e-9).argmin()
    max = (x[:, 0] - x[:, 1] * 1e-9).argmax()
    x = x[x[:, 1] <= (x[:, 0] - x[min, 0]) / (x[max, 0] - x[min, 0]) * (x[max, 1] - x[min, 1]) + x[min, 1]]
    return x[x[:, 0].argsort()]


y = [100 - i.metrics.get("a", 1 / 68) for i in a]
for j in "ihctauCrl":
    if j == "l":
        x = [sum(i["l"] for i in i.config["l"][:-1]) for i in a]
    else:
        x = [np.exp(sum((np.log(i[j]) + np.log(I[j])) * i.get("l", 1) for i, I in pk(i.config["l"])) / sum(i.get("l", 1) for i in i.config["l"][:-1]) / 2) for i in a]
    x = np.c_[x, y]
    plt.figure(figsize=(6, 1))
    plt.subplots_adjust(0, 0, 1, 1)
    plt.plot(*x.T, ".")
    x = alphashape(np.log(x), 0.8)
    for x in getattr(x, "geoms", [x]):
        plt.plot(*np.exp(lower(np.array(x.boundary.coords))).T, label=j)
    plt.yscale("log")
    plt.gca().yaxis.set_minor_formatter(plt.ScalarFormatter())
    plt.legend()
    plt.show()

In [ ]:
plt.plot(y, ".")
plt.yscale("log")
plt.gca().yaxis.set_minor_formatter(plt.ScalarFormatter())

import pandas as pd
def flat(a):
    return {i.split("/")[-1] + j: a for i, a in a.items() for j, a in flat(a).items()} if hasattr(a, "items") else {str(i) + j: a for i, a in enumerate(a) for j, a in flat(a).items()} if isinstance(a, list) else {"": a}
df = pd.DataFrame({"l": i.metrics["l"], "a": i.metrics["a"], **flat(i.config)} for i in a if "a" in i.metrics)

In [ ]:
from itertools import zip_longest

from IPython.display import clear_output


def str_model(a, m={Sequential: "|", Add: "+", SiLU: " "}):
    if isinstance(a, Sequential):
        return m.get(type(a), a._get_name()[0]) + "\n ".join(a.replace("\n", "\n ") for a in a for a in [str_model(a)] if a)
    if isinstance(a, Conv1d):
        return f"{a.in_channels} {a.out_channels}" + (f" {a.kernel_size[0]}" if a.kernel_size[0] > 1 else "")
    return m.get(type(a))


def catext(a, s=""):
    o = next(a)
    print(o)
    o = o.splitlines()
    for a in a:
        a = a.splitlines()
        l = max(map(len, o))
        o = [o.ljust(l) + s + a for o, a in zip_longest(o, a, fillvalue="")]
        clear_output(True)
        print("\n".join(o))


def catext2(a, s=""):
    o = next(a)
    print(o)
    o = o.splitlines()
    for a in a:
        a = a.splitlines()
        l = max((len(o) - len(a) + len(a.strip()) for o, a in zip(o, a)))
        o = [o.ljust(l + len(a) - len(a.strip())) + s + a.strip() for o, a in zip_longest(o, a, fillvalue="")]
        clear_output(True)
        print("\n".join(o))


catext2(f'{a.metrics["a"]:.0f}\n' + str_model(get_model(a.config)).replace("\n ", "\n")[1:] for a in sorted(a, key=lambda x: x.metrics.get("a", 0))[-10:])

In [ ]:
%reset -f in out

In [ ]:
import numpy as np
from tqdm import tqdm


def compress(A, N):
    m = A.max() + 1
    n = int(64 / np.log2(m))
    A = np.lib.stride_tricks.sliding_window_view(A, N)
    a = np.empty((len(A), (N / n).__ceil__()), "L")
    for i in range(a.shape[1]):
        t = A[:, i * n : i * n + n]
        np.matmul(t, m ** np.arange(t.shape[1], dtype="L"), a[:, i])
    return a.view(f"V{8*a.shape[1]}")[:, 0]


def count(a):
    t = np.zeros(a.max() + 1, "I")
    np.add.at(t, a, 1)
    return t[t.nonzero()]


def count2(a):
    a.sort()
    a = a[:-1] != a[1:]
    l = len(a)
    a = a.nonzero()[0].view("2I")[:, 0]
    return a[:1].copy() + 1, a[1:] - a[:-1], l - a[-1:].copy()


def entropy(a):
    l = sum(a.sum() for a in a)
    return sum(a @ np.log(a) for a in a for a in (np.divide(a, l, a.view("f")),))


a = np.fromfile("../dat/py.txt.map", "B")[np.memmap("../dat/py.txt")]
# try:
#     p = np.fromfile("../dat/py.txt.entropy")
# except FileNotFoundError:
#     p = np.zeros(0)
for i in tqdm(range(len(p), 40)):
    p = np.r_[p, entropy(count2(compress(a, i + 1)))]
    p.tofile("../dat/py.txt.entropy")
p

In [ ]:
import matplotlib.pyplot as plt

plt.plot(p)
plt.grid()

In [ ]:
plt.plot(np.exp(p - np.r_[0, p[:-1]]) * 100)
plt.grid()